# Clase 3 · Ponerle un freno a los coeficientes

**Machine Learning** · Programa de Especialización Data Scientist · Módulo 1 · Sesión 3 de 16

Docente: **Josef Renato Rodriguez**

---

## De la Clase 2 a la Clase 3

Seguimos con **California Housing** y el mismo objetivo: predecir el valor mediano de la vivienda.
La Clase 2 nos dejó dos problemas abiertos:

| Problema | Lo que vimos | Lo que hoy lo ataca |
|---|---|---|
| Coeficientes inestables | con 300 distritos, el coeficiente de `households` saltaba de −89 a +92 | **Ridge** |
| Demasiadas variables | 4 variables que miden casi lo mismo, VIF de 30 | **Lasso** |
| La recta no ve curvas | residuos con forma de embudo, patrón geográfico | **SVR** |

## Qué vas a hacer hoy

| Bloque | Contenido | Min |
|---|---|---|
| 1 | Ridge: encoger los coeficientes | 15 |
| 2 | Lasso: encoger y eliminar | 10 |
| 3 | Cuándo ayuda la regularización y cómo elegir α | 13 |
| 4 | SVR: una regresión que ignora los errores pequeños | 14 |

Igual que siempre: completa las celdas **`TU CÓDIGO`** y ejecuta la **`VERIFICACIÓN`** de abajo.

---
## Celda 0 · Preparación (idéntica a la Clase 2)

Misma carga, mismo corte 70/30 estratificado por zona, misma mediana para rellenar `total_bedrooms`
y mismo one-hot de `ocean_proximity`. Al final tienes `Xd_train`, `Xd_test`, `y_train`, `y_test`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

def metricas(y_real, y_pred):
    return {"R2": r2_score(y_real, y_pred),
            "MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_real, y_pred))}

URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
try:
    casas = pd.read_csv(URL)
except Exception:
    casas = pd.read_csv("california_housing.csv")

X_todo = casas.drop(columns=["median_house_value"])
y_todo = casas["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(
    X_todo, y_todo, test_size=0.3, random_state=SEED, stratify=casas["ocean_proximity"])

mediana_dorm = X_train["total_bedrooms"].median()
X_train, X_test = X_train.copy(), X_test.copy()
X_train["total_bedrooms"] = X_train["total_bedrooms"].fillna(mediana_dorm)
X_test["total_bedrooms"] = X_test["total_bedrooms"].fillna(mediana_dorm)

Xd_train = pd.get_dummies(X_train, columns=["ocean_proximity"], drop_first=True, dtype=int)
Xd_test = pd.get_dummies(X_test, columns=["ocean_proximity"], drop_first=True, dtype=int)
COLS = list(Xd_train.columns)

# El modelo de la Clase 2, para comparar
ols = LinearRegression().fit(Xd_train, y_train)
m_ols = metricas(y_test, ols.predict(Xd_test))
print(f"entrenamiento {Xd_train.shape} · prueba {Xd_test.shape}")
print(f"Regresión lineal de la Clase 2 → R² {m_ols['R2']:.4f} · MAE {m_ols['MAE']:,.0f}")

---
# Bloque 1 · Ridge: encoger los coeficientes  ·  15 min

La regresión lineal busca los coeficientes que hacen mínima la suma de errores al cuadrado. **Ridge**
añade un segundo término: un **castigo por tener coeficientes grandes**.

$$\text{Ridge minimiza} \quad \sum e_i^2 \; + \; \alpha \sum b_j^2$$

| Símbolo | Se lee | Qué es |
|---|---|---|
| $\sum e_i^2$ | «suma de e al cuadrado» | el error de siempre: qué tan bien ajusta |
| $\sum b_j^2$ | «suma de be jota al cuadrado» | el tamaño de los coeficientes (el intercepto no entra) |
| $\alpha$ | «alfa» | **la fuerza del freno**. Es un hiperparámetro: lo eliges tú |

- Con $\alpha = 0$ no hay freno: es la regresión lineal de la Clase 2.
- Con $\alpha$ enorme, el freno gana: todos los coeficientes van a 0 y el modelo predice el promedio.
- En medio, el modelo acepta **ajustar un poco peor** a cambio de **coeficientes más modestos y estables**.

### Ejercicio 1.1 — Ridge a mano

Volvemos a los 5 distritos de la Clase 2 ($\sum dx \cdot dy = 250$, $\sum dx^2 = 10$). Con una sola
variable, la pendiente de Ridge tiene fórmula cerrada: es la de mínimos cuadrados con **α sumado
al denominador**.

$$b_1^{\text{Ridge}} = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2 + \alpha}$$

Calcula la pendiente para α = 0, 10 y 40.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
x = np.array([2, 3, 4, 5, 6])
y = np.array([120, 150, 160, 200, 220])
dx, dy = x - x.mean(), y - y.mean()

b_alfa0 = np.sum(dx * dy) / (np.sum(dx**2) + 0)
b_alfa10 = np.sum(dx * dy) / (np.sum(dx**2) + 10)
b_alfa40 = np.sum(dx * dy) / (np.sum(dx**2) + 40)

for a, b in [(0, b_alfa0), (10, b_alfa10), (40, b_alfa40)]:
    print(f"α = {a:>2}  →  b1 = 250 / (10 + {a:>2}) = {b:>5.1f}")
print()
print("Con α = 10 la pendiente se reduce a la mitad; con α = 40, a la quinta parte.")
print("Nunca llega a 0 exactamente: solo se acerca. Esa es la marca de Ridge.")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("pendiente con α = 0", b_alfa0, 25),
     check("pendiente con α = 10", b_alfa10, 12.5),
     check("pendiente con α = 40", b_alfa40, 5)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Antes de usar Ridge: escalar es obligatorio

El castigo suma los coeficientes al cuadrado, y **el tamaño de un coeficiente depende de la unidad
de su variable**. En la Clase 2, `median_income` tenía un coeficiente de ~40 000 (por cada 10 mil USD)
y `total_rooms` uno de ~6 (por habitación). Sin escalar, Ridge castigaría casi solo al ingreso, no
porque importe menos, sino porque se mide en una unidad más grande.

Por eso Ridge (y Lasso, y SVR) siempre van con `StandardScaler` **dentro de un pipeline**: así el
escalado se aprende con el entrenamiento y se aplica igual a la prueba.

### Ejercicio 1.2 — Ridge con los datos reales

Entrena `make_pipeline(StandardScaler(), Ridge(alpha=10))` con `Xd_train` y evalúa en prueba.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
ridge = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xd_train, y_train)
m_ridge = metricas(y_test, ridge.predict(Xd_test))

print(f"{'':>20} | {'R²':>7} | {'MAE':>8}")
print(f"{'lineal (Clase 2)':>20} | {m_ols['R2']:>7.4f} | {m_ols['MAE']:>8,.0f}")
print(f"{'Ridge α = 10':>20} | {m_ridge['R2']:>7.4f} | {m_ridge['MAE']:>8,.0f}")
print()
print("Prácticamente iguales. ¿Entonces Ridge no sirve? Sigue leyendo.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("R² de Ridge en prueba", None if m_ridge is None else m_ridge["R2"], 0.6514),
     check("MAE de Ridge en prueba", None if m_ridge is None else m_ridge["MAE"], 49653.1540, tol=1)]
print()
print("1.2 OK" if all(r) else "Revisa 1.2")

### Demostración 1.3 — El camino de Ridge

¿Qué les pasa a los coeficientes al subir α de 0.1 a 100 000? (Coeficientes estandarizados: todos
en la misma escala).

In [ ]:
# ── DEMOSTRACIÓN 1.3 ─────────────────────────────────────────────────────
alfas = np.logspace(-1, 5, 25)
camino = []
for a in alfas:
    camino.append(make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(Xd_train, y_train)[-1].coef_)
camino = np.array(camino)

fig, ax = plt.subplots(figsize=(8, 4.5))
destacar = {"median_income": MAG, "total_bedrooms": BLUE, "households": GREEN, "population": NAVY}
for j, c in enumerate(COLS):
    ax.plot(alfas, camino[:, j], color=destacar.get(c, "#C5CEDF"), lw=2.2 if c in destacar else 1,
            label=c if c in destacar else None)
ax.axhline(0, color=GRAY, lw=1)
ax.axvline(10, color=GRAY, ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("α (escala logarítmica) →  más freno")
ax.set_ylabel("coeficiente estandarizado")
ax.set_title("Camino de Ridge: todos se encogen hacia cero, ninguno llega")
ax.legend(); plt.tight_layout(); plt.show()

Con α pequeño (izquierda) los coeficientes son los de la Clase 2. Al aumentar α, todos se
encogen, y los de las variables correlacionadas entre sí (dormitorios, población, y también latitud
con longitud) se encogen bastante antes que el del ingreso.
La línea punteada es α = 10: casi no se ha movido nada. Con 14 448 distritos, el modelo tiene datos
de sobra y apenas necesita freno.

¿Dónde se nota entonces? Recuerda la Clase 2: el problema aparecía con **muestras pequeñas**.

### Demostración 1.4 — Ridge estabiliza los coeficientes

Las mismas 5 muestras de 300 distritos de la Clase 2. Comparamos el coeficiente de `households` y de
`total_bedrooms` (en dólares por unidad) con regresión lineal y con Ridge α = 10.

In [ ]:
# ── DEMOSTRACIÓN 1.4 ─────────────────────────────────────────────────────
print(f"{'muestra':>7} || {'lineal: hogares':>15} | {'dormitorios':>11} || {'Ridge: hogares':>14} | {'dormitorios':>11}")
print("-" * 72)
for k in range(5):
    idx = np.random.default_rng(100 + k).choice(len(Xd_train), 300, replace=False)
    Xm, ym = Xd_train.iloc[idx], y_train.iloc[idx]
    lin = dict(zip(COLS, LinearRegression().fit(Xm, ym).coef_))
    rp = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(Xm, ym)
    rid = dict(zip(COLS, rp[-1].coef_ / rp[0].scale_))      # devuelto a dólares por unidad
    print(f"{k+1:>7} || {lin['households']:>15.0f} | {lin['total_bedrooms']:>11.0f} || {rid['households']:>14.0f} | {rid['total_bedrooms']:>11.0f}")

- Con regresión lineal, el coeficiente de hogares **cambia de signo** (-89 a +92).
- Con Ridge se mantiene **siempre positivo** (18 a 61), y el de dormitorios
  oscila mucho menos.

> Ridge no elimina la multicolinealidad: **la domestica**. Reparte el efecto entre las variables
> correlacionadas en lugar de dejar que una se dispare hacia arriba y la otra hacia abajo.

---
# Bloque 2 · Lasso: encoger y eliminar  ·  10 min

**Lasso** cambia una sola cosa: castiga el **valor absoluto** de los coeficientes en vez de su cuadrado.

$$\text{Lasso minimiza} \quad \sum e_i^2 \; + \; \alpha \sum \lvert b_j \rvert$$

Parece un detalle, pero tiene una consecuencia enorme: **Lasso lleva coeficientes a exactamente cero.**
Es decir, **elimina variables** por su cuenta. Ridge las encoge; Lasso además selecciona.

### Ejercicio 2.1 — ¿Qué variables elimina Lasso?

Entrena `make_pipeline(StandardScaler(), Lasso(alpha=1000, max_iter=100000))` y cuenta cuántos
coeficientes quedaron en cero. Los coeficientes están en `modelo[-1].coef_`.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
lasso = make_pipeline(StandardScaler(), Lasso(alpha=1000, max_iter=100000)).fit(Xd_train, y_train)
coef = lasso[-1].coef_
n_cero = int((coef == 0).sum())
vars_cero = [c for c, b in zip(COLS, coef) if b == 0]
m_lasso = metricas(y_test, lasso.predict(Xd_test))

print(f"Lasso eliminó {n_cero} de {len(COLS)} variables: {vars_cero}\n")
print(f"{'':>18} | {'variables':>9} | {'R²':>7} | {'MAE':>8}")
print(f"{'lineal':>18} | {len(COLS):>9} | {m_ols['R2']:>7.4f} | {m_ols['MAE']:>8,.0f}")
print(f"{'Lasso α = 1000':>18} | {len(COLS) - n_cero:>9} | {m_lasso['R2']:>7.4f} | {m_lasso['MAE']:>8,.0f}")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("variables eliminadas", n_cero, 2, tol=0),
     check_bool("eliminó total_rooms", vars_cero is not None and "total_rooms" in vars_cero),
     check("R² de Lasso en prueba", None if m_lasso is None else m_lasso["R2"], 0.6466)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

**Fíjate en cuál eliminó**: `total_rooms`, una de las cuatro variables de tamaño. Entre variables muy
correlacionadas, Lasso tiende a **quedarse con una y apagar las otras**. A cambio, el R² apenas baja
(0.6513 → 0.6466).

### Demostración 2.2 — El camino de Lasso: el orden en que se apagan las variables

In [ ]:
# ── DEMOSTRACIÓN 2.2 ─────────────────────────────────────────────────────
alfas_l = np.logspace(1, 4.8, 25)
camino_l, se_apaga = [], {}
for a in alfas_l:
    c_ = make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=100000)).fit(Xd_train, y_train)[-1].coef_
    camino_l.append(c_)
    for nombre, b in zip(COLS, c_):
        if b == 0 and nombre not in se_apaga:
            se_apaga[nombre] = a
camino_l = np.array(camino_l)

fig, ax = plt.subplots(figsize=(8, 4.5))
destacar = {"median_income": MAG, "total_bedrooms": BLUE, "households": GREEN, "total_rooms": NAVY}
for j, c in enumerate(COLS):
    ax.plot(alfas_l, camino_l[:, j], color=destacar.get(c, "#C5CEDF"), lw=2.2 if c in destacar else 1,
            label=c if c in destacar else None)
ax.axhline(0, color=GRAY, lw=1); ax.axvline(1000, color=GRAY, ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("α (escala logarítmica) →  más freno")
ax.set_ylabel("coeficiente estandarizado"); ax.set_title("Camino de Lasso: las variables se apagan una a una")
ax.legend(); plt.tight_layout(); plt.show()

print("Orden en que Lasso apaga las variables:")
for nombre, a in sorted(se_apaga.items(), key=lambda t: t[1]):
    print(f"  α ≈ {a:>8,.0f}  →  {nombre}")
print("  (median_income no se apaga nunca en este rango: es la más importante)")

### Ejercicio 2.3 — Un Lasso muy exigente

Repite con `alpha=5000`. ¿Cuántas variables sobreviven, y cuánto R² se pierde?

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
lasso_5000 = make_pipeline(StandardScaler(), Lasso(alpha=5000, max_iter=100000)).fit(Xd_train, y_train)
n_vivas = int((lasso_5000[-1].coef_ != 0).sum())
r2_5000 = r2_score(y_test, lasso_5000.predict(Xd_test))

print(f"Sobreviven {n_vivas} variables:")
print("  ", [c for c, b in zip(COLS, lasso_5000[-1].coef_) if b != 0])
print(f"R²: {m_ols['R2']:.4f} con 12 variables → {r2_5000:.4f} con {n_vivas}")
print()
print("La mitad de las variables explica casi todo. Ese es el trade-off que Lasso pone sobre la mesa:")
print("¿cuánta precisión estás dispuesto a perder por un modelo más simple de explicar?")

In [ ]:
# ── VERIFICACIÓN 2.3 ─────────────────────────────────────────────────────
r = [check("variables que sobreviven", n_vivas, 6, tol=0),
     check("R² con α = 5000", r2_5000, 0.6040)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.3")

| | Ridge | Lasso | Elastic Net |
|---|---|---|---|
| Castigo | $\sum b_j^2$ | $\sum \lvert b_j \rvert$ | una mezcla de los dos |
| ¿Lleva coeficientes a cero? | no, solo los encoge | **sí** | sí |
| Con variables correlacionadas | reparte el efecto entre ellas | se queda con una | reparte y selecciona |
| Úsalo cuando | todas las variables aportan algo | sospechas que muchas sobran | hay grupos de variables correlacionadas |

Elastic Net está en scikit-learn como `ElasticNet(alpha=..., l1_ratio=...)`: `l1_ratio=1` es Lasso
puro y `l1_ratio=0` es Ridge puro.

---
# Bloque 3 · Cuándo ayuda la regularización, y cómo elegir α  ·  13 min

Con los 14 448 distritos de entrenamiento, Ridge y Lasso empataron con la regresión
lineal en precisión. No es casualidad: con **muchos datos y pocas variables** (14 448 filas, 12
columnas), la regresión lineal no sobreajusta, así que no hay nada que frenar.

La regularización brilla en el caso opuesto: **pocos datos y muchas variables**. Lo fabricamos:

- Nos quedamos con **500 distritos** de entrenamiento.
- Creamos **todas las combinaciones de a dos** entre las 12 variables (ingreso × latitud,
  edad², etc.) con `PolynomialFeatures(2)`. Salen **90 variables** para 500 filas.

### Ejercicio 3.1 — La regresión lineal se pasa de lista

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
idx = np.random.default_rng(SEED).choice(len(Xd_train), 500, replace=False)
X_chico, y_chico = Xd_train.iloc[idx], y_train.iloc[idx]

def con_poli(modelo):
    """Escala, crea las 90 variables de grado 2, vuelve a escalar y aplica el modelo."""
    return make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False),
                         StandardScaler(), modelo)

lineal_poli = con_poli(LinearRegression()).fit(X_chico, y_chico)
r2_train_lin = r2_score(y_chico, lineal_poli.predict(X_chico))
r2_test_lin = r2_score(y_test, lineal_poli.predict(Xd_test))

print(f"{lineal_poli[2].n_features_in_} → {lineal_poli[-1].n_features_in_} variables para {len(X_chico)} distritos\n")
print(f"R² entrenamiento = {r2_train_lin:.3f}")
print(f"R² prueba        = {r2_test_lin:.3f}   ← la brecha de la Clase 1: overfitting")

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("R² en entrenamiento", r2_train_lin, 0.7869),
     check("R² en prueba", r2_test_lin, 0.5308)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

### ¿Cómo se elige α sin hacer trampa? Validación cruzada

En la Clase 1 elegimos la profundidad del árbol con un conjunto de **validación**, sin tocar la prueba.
Con solo 500 distritos, apartar otros 100 para validar duele. La alternativa es la **validación
cruzada con k particiones** (*k-fold*):

| Vuelta | Partición 1 | Partición 2 | Partición 3 | Partición 4 | Partición 5 |
|---|---|---|---|---|---|
| 1 | **valida** | entrena | entrena | entrena | entrena |
| 2 | entrena | **valida** | entrena | entrena | entrena |
| … | | | … | | |
| 5 | entrena | entrena | entrena | entrena | **valida** |

Cada distrito valida **una vez** y entrena **cuatro**. Se promedian los 5 resultados. La prueba no se
toca. (La veremos a fondo en la Clase 13; hoy basta con usarla.)

### Ejercicio 3.2 — Elegir α con validación cruzada

Para cada α, calcula el R² promedio de `cross_val_score(..., cv=cv, scoring="r2")` sobre
`X_chico`, `y_chico`. Quédate con el mejor y, recién entonces, mídelo **una vez** en prueba.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
alfas_cv = [0.1, 1, 10, 100, 1000]
r2_cv = []
for a in alfas_cv:
    puntajes = cross_val_score(con_poli(Ridge(alpha=a)), X_chico, y_chico, cv=cv, scoring="r2")
    r2_cv.append(puntajes.mean())
    print(f"α = {a:>6}  →  R² por partición {np.round(puntajes, 3)}  →  promedio {puntajes.mean():.3f}")

mejor_alfa = alfas_cv[int(np.argmax(r2_cv))]
ridge_poli = con_poli(Ridge(alpha=mejor_alfa)).fit(X_chico, y_chico)
r2_test_ridge = r2_score(y_test, ridge_poli.predict(Xd_test))

print(f"\nMejor α por validación cruzada: {mejor_alfa}")
print(f"{'':>24} | {'R² entrena':>10} | {'R² prueba':>9}")
print(f"{'lineal, 90 variables':>24} | {r2_train_lin:>10.3f} | {r2_test_lin:>9.3f}")
print(f"{'Ridge, 90 variables':>24} | {r2_score(y_chico, ridge_poli.predict(X_chico)):>10.3f} | {r2_test_ridge:>9.3f}")

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
ok = None not in r2_cv
r = [check_bool("calculaste los 5 promedios", ok, "r2_cv.append(puntajes.mean())"),
     check("mejor α", mejor_alfa, 10, tol=0),
     check("R² promedio de validación cruzada del mejor α", max(r2_cv) if ok else None, 0.6580),
     check("R² en prueba de Ridge", r2_test_ridge, 0.6572)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.2")

Ridge sacrifica un poco de ajuste en entrenamiento (0.787 → 0.742) y gana mucho en prueba
(0.531 → 0.657). Es la misma curva de la Clase 1: α es el control de complejidad.

- **α muy chico** (0.1): casi sin freno → overfitting → R² de validación bajo.
- **α muy grande** (1 000): freno excesivo → underfitting → R² de validación bajo.
- **En medio** (10): el punto dulce.

> **Cuándo vale la pena regularizar:** muchas variables respecto a las filas, variables muy
> correlacionadas, o cuando necesitas coeficientes estables para explicarlos. Con muchos datos y
> pocas variables, casi no cambia nada… y eso también es información útil.

---
# Bloque 4 · SVR: una regresión que ignora los errores pequeños  ·  14 min

La regresión lineal castiga **todos** los errores, hasta los de un dólar. La **regresión con vectores
de soporte** (SVR) parte de otra idea: traza un **tubo de ancho ±ε** alrededor de la predicción y
**ignora todo error que quede dentro del tubo**. Solo castiga lo que se sale, y lo hace en línea recta
(como el MAE), no al cuadrado.

| Hiperparámetro | Se lee | Qué controla |
|---|---|---|
| $\varepsilon$ | «épsilon» | el ancho del tubo: errores menores a ε no cuentan |
| $C$ | «ce» | cuánto importa salirse del tubo. C grande = el modelo se esfuerza más por ajustar |
| `kernel` | «kérnel» | la forma: `"linear"` traza un plano; `"rbf"` puede **curvarse** |

### Ejercicio 4.1 — La pérdida ε-insensible a mano

Con ε = 20 (miles de USD), la pérdida de cada residuo es $\max(0,\; \lvert e \rvert - \varepsilon)$.
Calcúlala para estos 4 residuos y compárala con la pérdida al cuadrado.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
residuos = np.array([5, -15, 30, -50])     # en miles de USD
eps = 20

perdida_eps = np.maximum(0, np.abs(residuos) - eps)
total_eps = perdida_eps.sum()
total_cuad = (residuos ** 2).sum()

print(f"{'residuo':>8} | {'ε-insensible':>12} | {'al cuadrado':>11}")
for e, p in zip(residuos, perdida_eps):
    print(f"{e:>8} | {p:>12} | {e**2:>11}")
print(f"{'total':>8} | {total_eps:>12} | {total_cuad:>11}")
print()
print("Los dos primeros quedan dentro del tubo: no cuentan nada.")
print("Y el de -50 pesa 30, no 2 500. Un distrito topado o un atípico no arrastra todo el modelo.")

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("pérdida ε-insensible total", total_eps, 40),
     check("pérdida al cuadrado total", total_cuad, 3650)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — SVR con kernel RBF

Dos detalles prácticos:

- **SVR es lento con muchos datos**: el tiempo crece más o menos con el cuadrado de las filas. Entrenamos
  con una muestra de **5 000 distritos** (unos segundos). Con los 14 448 tardaría varias veces más.
- ε se mide en la unidad de $y$. Para que «ε = 20» signifique **20 mil USD**, trabajamos con `y` en
  miles y multiplicamos la predicción por 1 000 al final.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
muestra = np.random.default_rng(SEED).choice(len(Xd_train), 5000, replace=False)
X_svr = Xd_train.iloc[muestra]
y_svr = y_train.iloc[muestra] / 1000          # en miles de USD

svr_rbf = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=100, epsilon=20)).fit(X_svr, y_svr)
pred_svr = svr_rbf.predict(Xd_test) * 1000
m_svr = metricas(y_test, pred_svr)

n_sv = len(svr_rbf[-1].support_)
print(f"Vectores de soporte: {n_sv:,} de {len(X_svr):,} distritos ({100*n_sv/len(X_svr):.0f} %)")
print("Son los distritos que quedaron fuera del tubo o en su borde: los únicos que definen el modelo.\n")
print(f"{'':>20} | {'R²':>7} | {'MAE':>8} | {'RMSE':>8}")
print(f"{'lineal (Clase 2)':>20} | {m_ols['R2']:>7.4f} | {m_ols['MAE']:>8,.0f} | {m_ols['RMSE']:>8,.0f}")
print(f"{'SVR RBF':>20} | {m_svr['R2']:>7.4f} | {m_svr['MAE']:>8,.0f} | {m_svr['RMSE']:>8,.0f}")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("R² del SVR en prueba", None if m_svr is None else m_svr["R2"], 0.7403, tol=2e-3),
     check("MAE del SVR en prueba", None if m_svr is None else m_svr["MAE"], 39820.3337, tol=100)]
print()
print("4.2 OK" if all(r) else "Revisa 4.2")

### Demostración 4.3 — ¿El tubo o la curva?

El SVR RBF tiene dos ventajas sobre la recta: el tubo (ignora errores pequeños y no eleva al cuadrado)
y la curvatura (el kernel). ¿Cuál pesa más? Separémoslas: un SVR **lineal** tiene el tubo pero no la
curva.

In [ ]:
# ── DEMOSTRACIÓN 4.3 ─────────────────────────────────────────────────────
if m_svr is not None:
    svr_lin = make_pipeline(StandardScaler(), SVR(kernel="linear", C=100, epsilon=20)).fit(X_svr, y_svr)
    m_svr_lin = metricas(y_test, svr_lin.predict(Xd_test) * 1000)
    ols_5000 = LinearRegression().fit(X_svr, y_svr * 1000)
    m_ols_5000 = metricas(y_test, ols_5000.predict(Xd_test))
    print("Todos entrenados con los mismos 5 000 distritos:\n")
    print(f"{'':>18} | {'tubo':>4} | {'curva':>5} | {'R²':>6} | {'MAE':>7} | {'RMSE':>7}")
    for nombre, tubo, curva, m in [("lineal", "no", "no", m_ols_5000), ("SVR lineal", "sí", "no", m_svr_lin), ("SVR RBF", "sí", "sí", m_svr)]:
        print(f"{nombre:>18} | {tubo:>4} | {curva:>5} | {m['R2']:>6.3f} | {m['MAE']:>7,.0f} | {m['RMSE']:>7,.0f}")

- **Solo el tubo** (SVR lineal) baja el MAE pero **sube** el RMSE: como no eleva al cuadrado, se
  preocupa menos por los errores grandes. Es la discusión MAE vs RMSE de la Clase 2, convertida en modelo.
- **El tubo y la curva** (SVR RBF) mejoran todo. La ganancia grande viene de la **curvatura**: el
  valor de una vivienda no depende de la latitud en línea recta, sino de en qué zona exacta está.

### Demostración 4.4 — Elegir C y ε sin tocar la prueba

Probamos 9 combinaciones. Para elegir usamos **otros 2 000 distritos del entrenamiento** que el SVR no
vio (validación). Tarda unos 20 segundos.

In [ ]:
# ── DEMOSTRACIÓN 4.4 ─────────────────────────────────────────────────────
if X_svr is not None:
    resto = np.setdiff1d(np.arange(len(Xd_train)), muestra)
    val = np.random.default_rng(SEED).choice(resto, 2000, replace=False)
    X_val, y_val = Xd_train.iloc[val], y_train.iloc[val]

    resultados = []
    print(f"{'C':>5} | {'ε':>3} | {'R² valid.':>9} | {'MAE valid.':>10} | {'vectores de soporte':>19}")
    for C in [10, 100, 1000]:
        for e in [5, 20, 50]:
            m = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=C, epsilon=e)).fit(X_svr, y_svr)
            pv = m.predict(X_val) * 1000
            resultados.append((r2_score(y_val, pv), C, e))
            print(f"{C:>5} | {e:>3} | {r2_score(y_val, pv):>9.4f} | {mean_absolute_error(y_val, pv):>10,.0f} | {len(m[-1].support_):>19,}")

    _, C_mejor, e_mejor = max(resultados)
    final = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=C_mejor, epsilon=e_mejor)).fit(X_svr, y_svr)
    m_final = metricas(y_test, final.predict(Xd_test) * 1000)
    print(f"\nElegido por validación: C = {C_mejor}, ε = {e_mejor}")
    print(f"En PRUEBA (una sola vez): R² {m_final['R2']:.4f} · MAE {m_final['MAE']:,.0f}")

Dos patrones para quedarse:

- **C más grande → mejor ajuste** en este rango. El modelo todavía tiene margen para curvarse más.
- **ε más grande → menos vectores de soporte.** Con un tubo ancho, más distritos quedan dentro y no
  cuentan. El modelo se vuelve más simple, pero si el tubo es demasiado ancho ignora información útil.

### Demostración 4.5 — Dónde se equivoca cada modelo

Los residuos sobre el mapa de California. Rojo: el modelo se quedó corto. Azul: se pasó.

In [ ]:
# ── DEMOSTRACIÓN 4.5 ─────────────────────────────────────────────────────
if pred_svr is not None:
    res_ols = y_test.values - ols.predict(Xd_test)
    res_svr = y_test.values - pred_svr
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    for ax, res, titulo in [(axes[0], res_ols, "Regresión lineal"), (axes[1], res_svr, "SVR RBF")]:
        sc = ax.scatter(X_test["longitude"], X_test["latitude"], c=np.clip(res, -150000, 150000),
                        cmap="coolwarm", s=4, vmin=-150000, vmax=150000)
        ax.set_title(f"{titulo} · MAE {mean_absolute_error(y_test, y_test.values - res):,.0f}")
        ax.set_xlabel("longitud")
    axes[0].set_ylabel("latitud")
    fig.colorbar(sc, ax=axes, label="residuo (USD)", shrink=0.8)
    plt.show()

En la regresión lineal se ven **manchas**: zonas enteras donde casi todos los puntos tienen el mismo
color (por ejemplo, el extremo norte del estado en azul, o partes del valle central en rojo). Eso es
la violación de **independencia** de la Clase 2: los errores de distritos vecinos se parecen. El SVR
RBF, al poder curvarse, mezcla mucho más los colores. Lo que sigue en rojo en los dos (parte de la
costa de Los Ángeles) son errores que ninguno de los dos modelos logra corregir.

> **El costo:** SVR no tiene coeficientes que se puedan leer. Ganaste precisión y perdiste la frase
> «por cada 10 mil de ingreso, la vivienda vale tanto más». En riesgos, ese trade-off se discute siempre.

---
# Cierre

### Los números del día (en prueba)

| Modelo | Entrenado con | Variables | R² | MAE |
|---|---|---|---|---|
| Regresión lineal (Clase 2) | 14 448 | 12 | 0.651 | 49 664 |
| Ridge α = 10 | 14 448 | 12 | 0.651 | 49 653 |
| Lasso α = 1 000 | 14 448 | **10** | 0.647 | 49 973 |
| Lineal con 90 variables | 500 | 90 | 0.531 | 49 787 |
| Ridge con 90 variables (α por CV) | 500 | 90 | 0.657 | 48 262 |
| SVR lineal | 5 000 | 12 | 0.642 | 48 002 |
| SVR RBF (C = 100, ε = 20) | 5 000 | 12 | 0.740 | 39 820 |
| **SVR RBF (C = 1000, ε = 20, por validación)** | 5 000 | 12 | **0.759** | **37 926** |

### Checklist de salida

- [ ] Sé qué castiga Ridge y qué castiga Lasso, y por qué Lasso elimina variables.
- [ ] Siempre escalo antes de regularizar (o de usar SVR), dentro de un pipeline.
- [ ] Sé que la regularización ayuda con muchas variables y pocos datos, y casi nada en el caso contrario.
- [ ] Elijo α (o C y ε) con validación o validación cruzada, nunca con la prueba.
- [ ] Sé qué hace el tubo ε de SVR y qué aporta el kernel RBF.
- [ ] Sé que más precisión a veces cuesta interpretabilidad.

### Próxima clase · Lunes 5/10

**Clase 4 · Árboles de regresión, Random Forest y XGBoost.** El SVR nos mostró que hay mucha
precisión escondida en las curvas. Los árboles la buscan de otra forma: partiendo el mapa en
regiones, y sin necesidad de escalar nada.